<a href="https://colab.research.google.com/github/152560/152560/blob/main/SwiggyDataSetAanlysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("srijanrawat86/swiggy-data-set")

print("Path to dataset files:", path)

100%|██████████| 251k/251k [00:00<00:00, 55.1MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/srijanrawat86/swiggy-data-set/versions/1


In [6]:
import os
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.appName("SwiggyDataAnalysis").getOrCreate()

In [7]:
# List files in the downloaded dataset directory
print(f"Files in {path}:\n{os.listdir(path)}")

Files in /root/.cache/kagglehub/datasets/srijanrawat86/swiggy-data-set/versions/1:
['swiggy.csv']


Assuming the primary dataset file is `swiggy.csv`, let's load it into a PySpark DataFrame.

In [8]:
# Construct the full path to the CSV file
csv_file_path = os.path.join(path, "swiggy.csv")

# Read the CSV file into a PySpark DataFrame
df_spark = spark.read.csv(csv_file_path, header=True, inferSchema=True)

# Display the first 5 rows and the schema
df_spark.show(5)
df_spark.printSchema()

+---+-----------+---------+-----------------+-----+-----------+-------------+--------------------+------------+-------------+
| ID|       Area|     City|       Restaurant|Price|Avg ratings|Total ratings|           Food type|     Address|Delivery time|
+---+-----------+---------+-----------------+-----+-----------+-------------+--------------------+------------+-------------+
|211|Koramangala|Bangalore|      Tandoor Hut|300.0|        4.4|          100|Biryani,Chinese,N...|   5Th Block|           59|
|221|Koramangala|Bangalore|    Tunday Kababi|300.0|        4.1|          100|    Mughlai,Lucknowi|   5Th Block|           56|
|246|  Jogupalya|Bangalore|          Kim Lee|650.0|        4.4|          100|             Chinese| Double Road|           50|
|248|Indiranagar|Bangalore|New Punjabi Hotel|250.0|        3.9|          500|North Indian,Punj...|80 Feet Road|           57|
|249|Indiranagar|Bangalore|              Nh8|350.0|        4.0|           50|Rajasthani,Gujara...|80 Feet Road|       

### 5 Questions for Data Transformation using PySpark on the Swiggy Dataset:

1.  **Extract and Categorize Food Types**: The 'Food type' column contains comma-separated values. Transform this column to create a new DataFrame where each row represents a unique `(Restaurant, Food Type)` pair, effectively unnesting the food types. Then, categorize these food types into broader categories (e.g., 'Indian', 'Chinese', 'Fast Food') and count how many restaurants offer each broad category.

2.  **Calculate Price Per Rating Point**: Create a new column called 'Price per Rating Point' by dividing the 'Price' by 'Avg ratings'. Handle potential division by zero or null values gracefully (e.g., by imputing or setting to null if ratings are zero/null). Analyze the distribution of this new metric.

3.  **Standardize 'Delivery time'**: The 'Delivery time' is given in minutes. Convert this to a categorical column representing 'Fast' (<=30 mins), 'Medium' (31-60 mins), and 'Slow' (>60 mins). Then, analyze the average ratings for each delivery time category.

4.  **Aggregate Restaurant Data by City and Area**: For each `(City, Area)` combination, calculate the following:
    *   Number of unique restaurants.
    *   Average 'Price'.
    *   Average 'Avg ratings'.
    *   Total 'Total ratings'.
    This transformation involves grouping and aggregation.

5.  **Identify Top 5 'Food type' for each City**: For each city, identify the top 5 most frequently offered food types. This will involve splitting the 'Food type' column, exploding it, grouping by city and food type, counting occurrences, and then ranking within each city to get the top 5.

## Question 1: Extract and Categorize Food Types

In [9]:
from pyspark.sql.functions import split, explode, trim, lower, col

# Split the 'Food type' column by comma and explode the array into new rows
df_food_types = df_spark.withColumn("Food_Type_Individual", explode(split(col("Food type"), ",")))

# Trim leading/trailing spaces and convert to lowercase for consistency
df_food_types = df_food_types.withColumn("Food_Type_Individual", trim(lower(col("Food_Type_Individual"))))

# Show the transformed DataFrame with individual food types
df_food_types.select("Restaurant", "Food_Type_Individual").show(10, truncate=False)

+-----------------+--------------------+
|Restaurant       |Food_Type_Individual|
+-----------------+--------------------+
|Tandoor Hut      |biryani             |
|Tandoor Hut      |chinese             |
|Tandoor Hut      |north indian        |
|Tandoor Hut      |south indian        |
|Tunday Kababi    |mughlai             |
|Tunday Kababi    |lucknowi            |
|Kim Lee          |chinese             |
|New Punjabi Hotel|north indian        |
|New Punjabi Hotel|punjabi             |
|New Punjabi Hotel|tandoor             |
+-----------------+--------------------+
only showing top 10 rows


In [10]:
from pyspark.sql.functions import split, explode, trim, lower, col

In [11]:
df_food_types = df_spark.withColumn("Food_Type_Individual", explode(split(col("Food type"), ",")))
df_food_types = df_food_types.withColumn("Food_Type_Individual", trim(lower(col("Food_Type_Individual"))))
df_food_types.select("Restaurant", "Food_Type_Individual").show(10, truncate=False)

+-----------------+--------------------+
|Restaurant       |Food_Type_Individual|
+-----------------+--------------------+
|Tandoor Hut      |biryani             |
|Tandoor Hut      |chinese             |
|Tandoor Hut      |north indian        |
|Tandoor Hut      |south indian        |
|Tunday Kababi    |mughlai             |
|Tunday Kababi    |lucknowi            |
|Kim Lee          |chinese             |
|New Punjabi Hotel|north indian        |
|New Punjabi Hotel|punjabi             |
|New Punjabi Hotel|tandoor             |
+-----------------+--------------------+
only showing top 10 rows


Calculate Price Per Rating Point

In [12]:
Price_Per_Rating_Point = df_food_types.withColumn("Price Per rating Point",col("Price")/col("Avg ratings"))
Price_Per_Rating_Point.select("Restaurant","Price Per rating Point").show(10)

+-----------------+----------------------+
|       Restaurant|Price Per rating Point|
+-----------------+----------------------+
|      Tandoor Hut|     68.18181818181817|
|      Tandoor Hut|     68.18181818181817|
|      Tandoor Hut|     68.18181818181817|
|      Tandoor Hut|     68.18181818181817|
|    Tunday Kababi|     73.17073170731707|
|    Tunday Kababi|     73.17073170731707|
|          Kim Lee|    147.72727272727272|
|New Punjabi Hotel|      64.1025641025641|
|New Punjabi Hotel|      64.1025641025641|
|New Punjabi Hotel|      64.1025641025641|
+-----------------+----------------------+
only showing top 10 rows


In [23]:
from pyspark.sql.functions import col, desc

Price_Per_Rating_Point.orderBy(col("Delivery time")).show()

+------+-----------+---------+--------------------+-----+-----------+-------------+--------------------+---------------+-------------+--------------------+----------------------+
|    ID|       Area|     City|          Restaurant|Price|Avg ratings|Total ratings|           Food type|        Address|Delivery time|Food_Type_Individual|Price Per rating Point|
+------+-----------+---------+--------------------+-----+-----------+-------------+--------------------+---------------+-------------+--------------------+----------------------+
| 10273|       Koti|Hyderabad|              Scoops|200.0|        4.5|          500|Ice Cream,Dessert...| Koti Main Road|           20|           ice cream|     44.44444444444444|
| 10273|       Koti|Hyderabad|              Scoops|200.0|        4.5|          500|Ice Cream,Dessert...| Koti Main Road|           20|            desserts|     44.44444444444444|
| 10273|       Koti|Hyderabad|              Scoops|200.0|        4.5|          500|Ice Cream,Dessert...| 

# Standardize **Delivery** time

In [25]:
df_delivery_category = Price_Per_Rating_Point.withColumn(
    "Delivery_Category",
    when(col("Delivery time") <= 30, "Fast")
    .when((col("Delivery time") > 30) & (col("Delivery time") <= 60), "Medium")
    .otherwise("Slow")
)

# Analyze the average ratings for each delivery time category
average_ratings_by_delivery = df_delivery_category.groupBy("Delivery_Category").agg(
    avg("Avg ratings").alias("Average_Ratings")
)

# Display the results
average_ratings_by_delivery.show()

+-----------------+------------------+
|Delivery_Category|   Average_Ratings|
+-----------------+------------------+
|             Fast| 4.126701570680636|
|           Medium|  3.74033283470497|
|             Slow|3.6563659720242443|
+-----------------+------------------+

